# Colok-Roll Tutorial: Exploratory Mode

This notebook walks through a single-image exploratory workflow:

1. Load an OME-TIFF
2. Inspect channels and metadata
3. Compare Z-slice strategies (exploratory selection)
4. Run background subtraction (with plots)
5. Run cell segmentation with the Cellpose Space API
6. Run colocalization (with mask plot)
7. Run puncta analysis (with elbow and detection plots)

> Update the paths, channel names, and channel selections for your dataset before running.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
import colokroll as cr

# ---- Update these paths ----
input_path = Path("/path/to/your/image.ome.tiff")
output_dir = Path("/path/to/output/exploratory")
output_dir.mkdir(parents=True, exist_ok=True)

# Optional: expected channel names for this dataset
channel_names = ["GM130", "Phalloidin", "Chlorotoxin", "DAPI"]

In [ ]:
loader = cr.ImageLoader(auto_convert=False)
image = loader.load_image(input_path)

print("Image shape (Z, Y, X, C):", image.shape)
pixel_size_um = loader.get_pixel_size()
print("Pixel size (um):", pixel_size_um)

if channel_names:
    loader.rename_channels(channel_names)

print("Channel names:", loader.get_channel_names())

In [ ]:
# Compare z-slice strategies (exploratory mode)
comparison = cr.compare_strategies(
    image,
    output_path=output_dir / "strategy_comparison",
    save_plots=True,
    display_inline=True,
    compute_quality=False,
)

print("Available strategies:", comparison.strategy_names)

# Pick your preferred strategy from the comparison
strategy_name = "FFT + Closest (Auto 0.8)"  # update if you prefer a different strategy
result = comparison.results[strategy_name]

filtered_image = image[result.indices_keep]
channel_names = loader.get_channel_names()

print(f"Selected strategy: {strategy_name}")
print(f"Filtered from {image.shape[0]} to {filtered_image.shape[0]} slices")

In [ ]:
# Background subtraction per channel
bg_subtractor = cr.BackgroundSubtractor()
channel_names = loader.get_channel_names()

bg_results = {}
for i, ch in enumerate(channel_names):
    ch_data = filtered_image[:, :, :, i]
    corrected, meta = bg_subtractor.subtract_background(
        image=ch_data,
        channel_name=ch,
        auto_cache_score_tolerance=0.20,
    )
    bg_results[ch] = (corrected, meta)
    print(f"{ch}: {meta.get('method', 'auto')}")

In [ ]:
# Plot background subtraction comparison for a quick visual check
fig = bg_subtractor.plot_background_subtraction_comparison(
    original_data=filtered_image,
    corrected_results=bg_results,
    channel_names=channel_names,
)
if fig is not None:
    plt.show()

In [ ]:
# Cell segmentation using the Cellpose Space API
# Requires gradio_client and an HF token if the space is gated.
segmenter = cr.CellSegmenter(output_dir=output_dir / "segmentation")

seg = segmenter.segment_from_results(
    results=bg_results,
    channel_a="Phalloidin",
    channel_b="DAPI",
    channel_weights=(0.8, 0.2),
    projection="mip",
    output_format="png8",
    save_basename=input_path.stem,
)

mask = seg.mask_array

print("Mask saved:", seg.mask_path)
print("Outlines saved:", seg.outlines_path)
print("Unique labels:", np.unique(mask).size - 1)

In [ ]:
# Colocalization (with mask plot)
corrected_stack = np.stack([bg_results[ch][0] for ch in channel_names], axis=-1)

coloc_mask_plot = output_dir / f"{input_path.stem}_coloc_mask.png"
coloc = cr.compute_colocalization(
    image=corrected_stack,
    mask=mask,
    channel_a="GM130",
    channel_b="Chlorotoxin",
    channel_names=channel_names,
    thresholding="otsu",
    min_threshold_sigma=3.0,
    min_area="auto",
    min_area_fraction=0.9,
    max_border_fraction=0.2,
    border_margin_px=1,
    drop_label_1=True,
    plot_mask=True,
    plot_mask_save=coloc_mask_plot,
)

if coloc_mask_plot.exists():
    display(Image(filename=str(coloc_mask_plot)))

print(f"Pearson r: {coloc['results']['total_image']['pearson_r']:.3f}")
print(f"Manders M1: {coloc['results']['total_image']['manders_m1']:.3f}")
print(f"Manders M2: {coloc['results']['total_image']['manders_m2']:.3f}")
print(f"Jaccard: {coloc['results']['total_image']['jaccard']:.3f}")

In [ ]:
# Puncta analysis (with elbow + detection plots)
puncta_channel = "Chlorotoxin"  # update for your puncta marker
puncta = cr.compute_puncta(
    image=corrected_stack,
    mask=mask,
    channel=puncta_channel,
    channel_names=channel_names,
    pixel_size_um=pixel_size_um,
    projection="mip",
    detection_method="bigfish",
    return_threshold_data=True,
)

cr.plot_puncta_elbow(puncta)
plt.show()

puncta_channel_index = channel_names.index(puncta_channel)
puncta_mip = corrected_stack[..., puncta_channel_index].max(axis=0)

fig = cr.plot_puncta_detection(
    puncta,
    puncta_mip,
    cell_mask=mask,
    title=f"{puncta_channel} puncta",
)
if fig is not None:
    plt.show()

print(f"Total puncta: {puncta['results']['total_image']['total_puncta_count']}")
print(f"Cells analyzed: {puncta['results']['summary']['cells_count']}")

In [ ]:
# Save a small summary for reference
summary_path = output_dir / f"{input_path.stem}_summary.txt"
summary_path.write_text(
    "\n".join(
        [
            f"input={input_path}",
            f"shape={image.shape}",
            f"kept_slices={len(result.indices_keep)}",
            f"z_strategy={strategy_name}",
            f"mask={seg.mask_path}",
            f"coloc_pearson={coloc['results']['total_image']['pearson_r']:.3f}",
            f"coloc_manders_m1={coloc['results']['total_image']['manders_m1']:.3f}",
            f"coloc_manders_m2={coloc['results']['total_image']['manders_m2']:.3f}",
            f"coloc_jaccard={coloc['results']['total_image']['jaccard']:.3f}",
            f"puncta_channel={puncta_channel}",
            f"puncta_total={puncta['results']['total_image']['total_puncta_count']}",
        ]
    )
)
print("Saved summary:", summary_path)